In [1]:
import pandas as pd
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory

In [2]:
largura_padrao = 4200
pedidos = ['pedido1','pedido2','pedido3']

demandas ={
    'pedido1':{
        'quantidade':100,
        'tamanho':1300,
    },
    'pedido2':{
        'quantidade':150,
        'tamanho':1400,
    },
    'pedido3':{
        'quantidade':120,
        'tamanho':1600,
    },
}
# Possibilidade 1 exemplo, 3 pedidos1 dão 3900, sobrando 300 de refugos (total de 4200)
# Na possibilida2 ja tenho que colocar quantidade de  2 no pedido1
possibilidades = {
    'possibilidade1': {
        'pedido1': 3,
        'pedido2': 0,
        'pedido3': 0,
        'refugos':300,
    },
    'possibilidade2': {
        'pedido1': 2,
        'pedido2': 1,
        'pedido3': 0,
        'refugos':200,
    },
    'possibilidade3': {
        'pedido1': 2,
        'pedido2': 0,
        'pedido3': 1,
        'refugos':0,
    },
    'possibilidade4': {
        'pedido1': 1,
        'pedido2': 2,
        'pedido3': 0,
        'refugos':100,
    },
    'possibilidade5': {
        'pedido1': 0,
        'pedido2': 0,
        'pedido3': 2,
        'refugos':1000,
    },
    'possibilidade6': {
        'pedido1': 0,
        'pedido2': 1,
        'pedido3': 1,
        'refugos':1200,
    },
    'possibilidade7': {
        'pedido1':0 ,
        'pedido2':3,
        'pedido3':0,
        'refugos':0,
    },

    
}



In [9]:
model = pyo.ConcreteModel()

model.possibilidades = pyo.Set(initialize=list(possibilidades.keys()))
model.pedidos = pyo.Set(initialize=pedidos)

model.x = pyo.Var(model.possibilidades, bounds=(0, None), domain=pyo.NonNegativeIntegers)

def objective_rule(model):
    refugos_total = sum(model.x[p] * possibilidades[p]['refugos'] for p in model.possibilidades)
    return  refugos_total

model.obj = pyo.Objective(rule=objective_rule, sense=pyo.minimize)

def demanda(model, pedidos):
    return sum(model.x[p] * possibilidades[p][pedidos] for p in model.possibilidades) == demandas[pedidos]['quantidade']
model.demanda_constraint = pyo.Constraint(model.pedidos, rule=demanda)

In [10]:
model.pprint()

2 Set Declarations
    pedidos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'pedido1', 'pedido2', 'pedido3'}
    possibilidades : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    7 : {'possibilidade1', 'possibilidade2', 'possibilidade3', 'possibilidade4', 'possibilidade5', 'possibilidade6', 'possibilidade7'}

1 Var Declarations
    x : Size=7, Index=possibilidades
        Key            : Lower : Value : Upper : Fixed : Stale : Domain
        possibilidade1 :     0 :  None :  None : False :  True : NonNegativeIntegers
        possibilidade2 :     0 :  None :  None : False :  True : NonNegativeIntegers
        possibilidade3 :     0 :  None :  None : False :  True : NonNegativeIntegers
        possibilidade4 :     0 :  None :  None : False :  True : NonNegativeIntegers
        possibilidade5 :     0 :  None :  None : False :  True : Non

In [11]:
opt = SolverFactory('cplex')
res = opt.solve(model, tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\joaon\AppData\Local\Temp\tmp87_kcxoz.cplex.log' open.
CPLEX> Problem 'C:\Users\joaon\AppData\Local\Temp\tmpguxs4ebh.pyomo.lp' read.
Read time = 0.02 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\joaon\AppData\Local\Temp\tmpguxs4ebh.pyomo.lp
Objective sense      : Minimize
Variables            :       7  [General Integer: 7]
Objective nonzeros   :       5
Linear constraints   :       3  [Equal: 3]
  Nonzeros           :      11
  RHS nonzeros       :       3

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 100.0000      

In [28]:
for m in model.x:
    print(f'{m}: {model.x[m].value} vezes')
    print(possibilidades[m])
print('-----------')
print(f'Valor mínimo de refugos: {model.obj()/1000} metros')

possibilidade1: -0.0 vezes
{'pedido1': 3, 'pedido2': 0, 'pedido3': 0, 'refugos': 300}
possibilidade2: -0.0 vezes
{'pedido1': 2, 'pedido2': 1, 'pedido3': 0, 'refugos': 200}
possibilidade3: 50.0 vezes
{'pedido1': 2, 'pedido2': 0, 'pedido3': 1, 'refugos': 0}
possibilidade4: -0.0 vezes
{'pedido1': 1, 'pedido2': 2, 'pedido3': 0, 'refugos': 100}
possibilidade5: 35.0 vezes
{'pedido1': 0, 'pedido2': 0, 'pedido3': 2, 'refugos': 1000}
possibilidade6: -0.0 vezes
{'pedido1': 0, 'pedido2': 1, 'pedido3': 1, 'refugos': 1200}
possibilidade7: 50.0 vezes
{'pedido1': 0, 'pedido2': 3, 'pedido3': 0, 'refugos': 0}
-----------
Valor mínimo de refugos: 35.0 metros


{'pedido1': 2, 'pedido2': 0, 'pedido3': 1, 'refugos': 0}
